# 01 — Dataset audit

Audit WikiANN language configurations: **metadata, split sizes, label schema,
IOB2 validity, and duplicates**. Everything here calls reusable functions from
`src/axner`; no core logic lives in this notebook.

**Requirements:** the first cell run of `axner.data.wikiann.audit_language`
downloads that language's parquet files (a few MB) into `data/hf_cache`
(git-ignored). After that it works offline.

For a full CLI audit of all research languages:

```
python scripts/audit_wikiann.py --languages id ms tl en jv su
```

Results are written to `data/audits/` and summarized in `docs/data_card.md`.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
print("repo root:", repo_root)

In [ ]:
from axner.constants import LABEL_NAMES
from axner.data.wikiann import audit_language

cache_dir = str(repo_root / "data" / "hf_cache")
report = audit_language("jv", cache_dir=cache_dir)
print({split: info["num_rows"] for split, info in report["splits"].items()})

In [ ]:
import pandas as pd

rows = []
for split, info in report["splits"].items():
    rows.append({
        "split": split,
        "rows": info["num_rows"],
        "iob2_invalid": info["iob2_scan"]["n_invalid_examples"],
        "duplicate_rows": info["within_split_duplicates"]["n_duplicate_rows"],
    })
pd.DataFrame(rows)

In [ ]:
# Label schema check: the dataset's ClassLabel names vs the canonical project list.
report["label_schema"]

In [ ]:
# Cross-split duplicate counts (content leakage between train/validation/test).
report["cross_split_duplicates"]

## Full audit (optional)

Verified split sizes (recorded 2026-11): jv and su have **100 examples per
split** — EXP-0's "100 target sentences" is the entire train split, and target
test sets hold 100 sentences each. See `docs/data_card.md` and
`configs/project.yaml` (`known_max_train_sizes`).

In [ ]:
RUN_FULL_AUDIT = False  # set True to audit every research language (downloads each once)

if RUN_FULL_AUDIT:
    from axner.data.wikiann import audit_languages

    full = audit_languages(["id", "ms", "tl", "en", "jv", "su"], cache_dir=cache_dir)
    pd.DataFrame(full["summary"])